# Pensieve ML - Phase 4: RAG Knowledge Grounding & Retrieval Layer

> **ETHICAL AND SCIENTIFIC NOTICE**:
> - Phase 4 contains **NO LLM** and performs **NO text or reflection generation**.
> - All knowledge entries are **descriptive reflective frameworks and philosophical concepts**.
> - Retrieval relevance reflects semantic similarity for reflective inquiry; it does **NOT** constitute a medical diagnosis, clinical conclusion, or psychiatric evaluation.
>
> **DATASET NOTICE**:
> The 20 concepts demonstrated below constitute a representative **DEVELOPMENT / TEST dataset** designed for pipeline testing, vector indexing verification, and retrieval benchmarking. This is **NOT** the final 54-concept production set.

In [ ]:
import sys
import os

# Ensure project root is on sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from ml.rag.documents import KnowledgeBase, ConceptDocument
from ml.rag.embeddings import ConceptEmbedder
from ml.rag.index import VectorIndex
from ml.rag.retriever import ConceptRetriever, PatternSignal, DeterministicQueryBuilder
from ml.rag.evaluate import evaluate_retrieval, RETRIEVAL_BENCHMARK

print("Phase 4 RAG modules imported successfully.")

## 1. Inspecting the Curated Knowledge Base (Development Dataset)
Each concept is structured with strict attribution and non-diagnostic cautions.

In [ ]:
kb = KnowledgeBase.load_development_dataset()
print(f"Loaded {len(kb)} development concepts.\n")

sample_doc = kb.get_document("stoic_dichotomy_of_control")
print(f"Concept Name: {sample_doc.name}")
print(f"Category:     {sample_doc.category}")
print(f"Definition:   {sample_doc.definition}")
print(f"Source:       {sample_doc.source}")
print(f"Cautions:     {sample_doc.cautions}")

## 2. Vector Indexing with FAISS (`IndexFlatIP`) and `all-MiniLM-L6-v2`

In [ ]:
retriever = ConceptRetriever(kb, default_top_k=3, default_threshold=0.25)
print(f"Index Backend: {'FAISS' if retriever.index.is_faiss_active() else 'NumPy'}")
print(f"Indexed Concepts: {retriever.index.size()}")
print(f"Embedding Dimension: {retriever.embedder.embedding_dim}")

## 3. Deterministic Query Construction from Phase 1–3 Pattern Signals (No LLM)

In [ ]:
signals = PatternSignal(
    emotions={"annoyance": 0.74, "nervousness": 0.62, "joy": 0.05},
    themes=[{"cluster_id": 0, "name": "Work & Engineering", "frequency": 0.55}],
    linguistic_patterns={"negation_ratio": 0.08, "question_count": 2, "first_person_pronoun_ratio": 0.18},
    longitudinal_patterns=[
        "Work-related theme recurring across 3 consecutive windows",
        "Annoyance increased significantly (+0.25) across recent entries",
        "Perceived imbalance between task demands and available recovery time",
    ],
)

builder = DeterministicQueryBuilder()
query, audit = builder.build_query(signals)
print(f"Synthesized Query: \"{query}\"\n")
print("Audit Trail:")
for k, v in audit.items():
    print(f"  {k}: {v}")

## 4. Top-K Semantic Retrieval with Similarity Threshold Gating

In [ ]:
response = retriever.retrieve(signals, top_k=3, similarity_threshold=0.25)
print(f"Retrieval Status: {response['status']}\n")

for rank, item in enumerate(response["results"], start=1):
    print(f"{rank}. {item['name']} ({item['category']})")
    print(f"   Similarity Score: {item['similarity_score']:.4f}")
    print(f"   Definition:       {item['definition']}")
    print(f"   Source:           {item['source']}")
    print(f"   Safety Cautions:  {item['cautions']}\n")

## 5. Out-of-Domain Abstention Control
Demonstrates that queries below the similarity threshold produce `no_relevant_concepts` rather than forcing an irrelevant concept.

In [ ]:
ood_query = "how to adjust carburetor idle mixture screws on a motorcycle engine"
ood_res = retriever.retrieve(ood_query, similarity_threshold=0.25)

print(f"Query: \"{ood_query}\"")
print(f"Status: {ood_res['status']}")
print(f"Message: {ood_res.get('message')}")
print(f"Results count: {len(ood_res['results'])}")

## 6. Retrieval Benchmark Evaluation
Measures Recall@K, Precision@K, and MRR against the curated evaluation benchmark.

In [ ]:
eval_results = evaluate_retrieval(retriever, benchmark=RETRIEVAL_BENCHMARK, similarity_threshold=0.25)
metrics = eval_results["summary_metrics"]

print("BENCHMARK SUMMARY METRICS:")
print(f"  Mean Reciprocal Rank (MRR) : {metrics['mrr']:.4f}")
print(f"  Hit@1                      : {metrics['mean_hit@1']:.4f}")
print(f"  Hit@3                      : {metrics['mean_hit@3']:.4f}")
print(f"  Recall@1                   : {metrics['mean_recall@1']:.4f}")
print(f"  Recall@3                   : {metrics['mean_recall@3']:.4f}")
print(f"  Recall@5                   : {metrics['mean_recall@5']:.4f}")
print(f"  Precision@1                : {metrics['mean_precision@1']:.4f}")
print(f"  Precision@3                : {metrics['mean_precision@3']:.4f}")
print(f"  Out-of-Domain Rejection    : {metrics['ood_rejection_rate'] * 100:.1f}%")